In [15]:
import wrds
import pandas as pd

pd.set_option("mode.copy_on_write", True)

db = wrds.Connection(wrds_username='ppizam')


Loading library list...
Done


In [16]:
vidas = db.raw_sql("""
    select permno, permco, namedt, nameenddt,
           ticker, comnam, ncusip, cusip,
           shrcd, exchcd, siccd, shrcls
    from crsp.stocknames
    where nameenddt >= '2020-01-01'
      and namedt   <= '2024-12-31'
      and exchcd in (1, 2, 3, 4)
    order by ticker, namedt
""", date_cols=['namedt', 'nameenddt'])
 
print(len(vidas), vidas.permno.nunique())


16828 11899


In [17]:
bajas = db.raw_sql("""
    select permno, dlstdt, dlstcd, dlret
    from crsp.msedelist
    where dlstdt between '2020-01-01' and '2024-12-31'
""", date_cols=['dlstdt'])


In [18]:
msf = db.raw_sql("""
    select permno, date, abs(prc) as precio,
           shrout, vol, ret
    from crsp.msf
    where date between '2020-01-01' and '2024-12-31'
""", date_cols=['date']).copy()

msf['market_cap_usd'] = msf['precio'] * msf['shrout'] * 1000


/tmp/34895627.1.jupyterhub.q/ipykernel_50701/3255917673.py:8: ChainedAssignmentError: A value is trying to be set on a copy of a DataFrame or Series through chained assignment.
When using the Copy-on-Write mode, such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy.

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msf['market_cap_usd'] = msf['precio'] * msf['shrout'] * 1000


In [19]:
print('market_cap_usd' in msf.columns)

True


In [14]:
msf['market_cap_usd'].describe()

count              537838.0
mean      6357498924.574066
std      50815386430.592323
min                 33468.0
25%             66584333.75
50%             326915925.0
75%            1869913522.5
max         3785304395660.0
Name: market_cap_usd, dtype: Float64

In [21]:
libs = db.list_libraries()
sorted([l for l in libs if 'ccm' in l or 'link' in l or 'wrdsapp' in l])

['contrib_patent_firm_link',
 'fjc_linking',
 'wrdsapps',
 'wrdsapps_eushort',
 'wrdsapps_evtstudy_int',
 'wrdsapps_evtstudy_us',
 'wrdsapps_finratio',
 'wrdsapps_link_comp_eushort',
 'wrdsapps_link_crsp_bond',
 'wrdsapps_link_crsp_factset',
 'wrdsapps_link_crsp_optionm',
 'wrdsapps_link_crsp_taq',
 'wrdsapps_patents',
 'wrdsapps_subsidiary',
 'wrdsapps_windices',
 'wrdsappssamp_all']

In [22]:
db.list_tables(library='wrdsapps_finratio')


['firm_ratio', 'id']

In [24]:
bridge = db.raw_sql("""
    select distinct permno, gvkey
    from wrdsapps_finratio.firm_ratio
    where public_date between '2020-01-01' and '2024-12-31'
""")
print(len(bridge), bridge.permno.nunique())

5007 4816


In [25]:
comp_info = db.raw_sql("""
    select gvkey, cik, gsector, ggroup, gind, conm
    from comp.company
""")

bridge = bridge.merge(comp_info, on='gvkey', how='left')
vidas_enr = vidas.merge(bridge, on='permno', how='left')

# Cobertura del cruce, para documentar:
print(vidas_enr['cik'].notna().mean())

0.4311070131965654


In [26]:
comunes = vidas_enr[vidas_enr['shrcd'].isin([10, 11])]
print(comunes['cik'].notna().mean())

0.8765773344549934


In [27]:
# Confirmar la duplicación:
print(len(vidas), len(vidas_enr))  # si difieren, hubo duplicados

# Ver los permnos con más de un gvkey:
dups = bridge[bridge.duplicated('permno', keep=False)].sort_values('permno')
print(dups.head(20))

# Deduplicar el puente a un gvkey por permno y rehacer el merge:
bridge_1 = bridge.sort_values(['permno', 'gvkey']).drop_duplicates('permno', keep='first')
vidas_enr = vidas.merge(bridge_1, on='permno', how='left')
print(len(vidas), len(vidas_enr))  # ahora deben ser iguales

16828 17353
      permno   gvkey         cik gsector ggroup    gind  \
4167   11992  015197  0000814184      40   4010  401010   
4899   11992  015363  0000019612      40   4010  401010   
2510   12350  040116  0001499961      25   2510  251020   
2001   12350  163806        <NA>      45   4510  451030   
2168   13111  186607        <NA>      35   3520  352010   
2431   13111  036333  0001126234      35   3520  352010   
2679   13743  008092  0001525221      10   1010  101010   
2825   13743  187766        <NA>      10   1010  101010   
4208   13883  034528        <NA>      35   3520  352010   
486    13883  039104  0001544227      35   3520  352010   
4756   13965  032984  0001173281      35   3520  352010   
4232   13965  160578        <NA>      35   3520  352010   
4083   14046  018161        <NA>      35   3520  352010   
1037   14046  036525  0001383701      35   3520  352010   
771    14094  170420        <NA>      35   3520  352010   
729    14094  038539  0001349929      35   3

In [31]:
multi = bridge[bridge.duplicated('permno', keep=False)]['permno'].unique()
vidas_enr = vidas_enr.assign(gvkey_multiple=vidas_enr['permno'].isin(multi))
print(vidas_enr['gvkey_multiple'].sum(), 'tramos con permno multi-gvkey')

525 tramos con permno multi-gvkey


In [32]:
import warnings
warnings.filterwarnings('ignore', category=pd.errors.ChainedAssignmentError)

In [40]:
import numpy as np

W0, W1 = pd.Timestamp('2020-01-01'), pd.Timestamp('2024-12-31')

# --- 6.1 Recorte de tramos a la ventana ---
v = vidas_enr.copy().sort_values(['permno', 'namedt'])
v = v.assign(
    fecha_inicio=v['namedt'].clip(lower=W0),
    fecha_fin=v['nameenddt'].clip(upper=W1),
)

# --- 6.2 Consolidar tramos contiguos del mismo permno-ticker ---
# (CRSP abre fila nueva ante cualquier cambio menor; aquí un "bloque" es
#  una vida: mismo permno y mismo ticker sin huecos)# --- 6.2 corregido: consolidar tramos contiguos del mismo permno-ticker ---
v = vidas_enr.copy().sort_values(['permno', 'namedt'])

prev_ticker = v.groupby('permno')['ticker'].shift()
prev_fin    = v.groupby('permno')['nameenddt'].shift()

cambio = (
    prev_ticker.isna()                                        # primer tramo de cada permno
    | (v['ticker'] != prev_ticker).fillna(False)              # cambio de ticker
    | (v['namedt'] > prev_fin + pd.Timedelta(days=7)).fillna(False)  # hueco > 7 dias
)
cambio = cambio.astype('bool')          # bool puro, sin NA

v = v.assign(
    fecha_inicio=v['namedt'].clip(lower=W0),
    fecha_fin=v['nameenddt'].clip(upper=W1),
    bloque=cambio.groupby(v['permno']).cumsum().astype(int),
)

vig = v.groupby(['permno', 'bloque'], as_index=False).agg(
    ticker=('ticker', 'last'),
    comnam=('comnam', 'last'),
    fecha_inicio=('fecha_inicio', 'min'),
    fecha_fin=('fecha_fin', 'max'),
    ncusip=('ncusip', 'last'),
    shrcd=('shrcd', 'last'),
    exchcd=('exchcd', 'last'),
    siccd=('siccd', 'last'),
    permco=('permco', 'last'),
    gvkey=('gvkey', 'last'),
    cik=('cik', 'last'),
    gsector=('gsector', 'last'),
    gvkey_multiple=('gvkey_multiple', 'max'),
)

# Verificacion inmediata: no debe perderse ningun permno
print(len(v), 'tramos ->', len(vig), 'vidas |',
      v['permno'].nunique(), 'permnos en origen,',
      vig['permno'].nunique(), 'en vig')
assert vig['permno'].nunique() == v['permno'].nunique(), 'se perdieron permnos'
# --- 6.3 Cruzar delistings ---
bajas_u = bajas.sort_values('dlstdt').drop_duplicates('permno', keep='last')
vig = vig.merge(bajas_u[['permno', 'dlstdt', 'dlstcd']], on='permno', how='left')

es_ultima = vig['fecha_fin'] == vig.groupby('permno')['fecha_fin'].transform('max')
es_baja = es_ultima & vig['dlstdt'].notna() & (vig['dlstcd'] >= 200)

vig = vig.assign(
    listing_status=np.where(es_baja, 'delisted', 'active'),
    delist_date=vig['dlstdt'].where(es_baja),
)
print(vig['listing_status'].value_counts())

# --- 6.4 Cambios de ticker (baja + alta enlazadas por permno) ---
vig = vig.sort_values(['permno', 'fecha_inicio'])
vig = vig.assign(
    ticker_previo=vig.groupby('permno')['ticker'].shift(),
    ticker_siguiente=vig.groupby('permno')['ticker'].shift(-1),
)
renombres = vig[vig['ticker_siguiente'].notna()]
print(len(renombres), 'vidas que terminan en cambio de ticker')

# --- 6.5 Símbolos reutilizados por otro emisor ---
vig = vig.assign(
    ticker_reutilizado=vig['ticker'].map(vig.groupby('ticker')['permno'].nunique()) > 1
)
print(vig['ticker_reutilizado'].sum(), 'vidas con ticker compartido entre permnos')

# --- 6.6 ipo_year: primera aparición del permno en toda la historia de CRSP ---
primeras = db.raw_sql("""
    select permno, min(namedt) as primera_fecha
    from crsp.stocknames
    group by permno
""", date_cols=['primera_fecha'])
vig = vig.merge(primeras, on='permno', how='left')
vig = vig.assign(ipo_year=vig['primera_fecha'].dt.year)

16828 tramos -> 12961 vidas | 11899 permnos en origen, 11899 en vig
listing_status
active      9962
delisted    2999
Name: count, dtype: int64
1062 vidas que terminan en cambio de ticker
491 vidas con ticker compartido entre permnos


In [41]:
vig = vig.assign(
    estado_vida=np.select(
        [vig['listing_status'].eq('delisted'),
         vig['ticker_siguiente'].notna()],
        ['delisted', 'renombrada'],
        default='active',
    )
)
print(vig['estado_vida'].value_counts())

estado_vida
active        8900
delisted      2999
renombrada    1062
Name: count, dtype: int64


In [42]:
anclas = ['GME', 'AMC', 'BBBY', 'FB', 'META', 'ABNB', 'COIN', 'HOOD',
          'RIVN', 'ARM', 'RDDT', 'DJT', 'SMCI', 'SIVB', 'SBNY']
cols = ['ticker', 'permno', 'comnam', 'fecha_inicio', 'fecha_fin',
        'listing_status', 'delist_date', 'ticker_previo', 'ticker_siguiente']
print(vig[vig['ticker'].isin(anclas)][cols]
      .sort_values(['ticker', 'fecha_inicio']).to_string(index=False))

ticker  permno                           comnam fecha_inicio  fecha_fin listing_status delist_date ticker_previo ticker_siguiente
  ABNB   20190                       AIRBNB INC   2020-12-10 2024-12-31         active         NaT          <NA>             <NA>
   AMC   14328 A M C ENTERTAINMENT HOLDINGS INC   2020-01-01 2024-12-31         active         NaT          <NA>             <NA>
   ARM   24252               A R M HOLDINGS PLC   2023-09-14 2024-12-31         active         NaT          <NA>             <NA>
  BBBY   77659            BED BATH & BEYOND INC   2020-01-01 2023-05-02       delisted  2023-05-02          <NA>             <NA>
  COIN   20892              COINBASE GLOBAL INC   2021-04-14 2024-12-31         active         NaT          <NA>             <NA>
   DJT   21927      TRUMP MEDIA & TECH GRP CORP   2024-03-26 2024-12-31         active         NaT          DWAC             <NA>
    FB   13407               META PLATFORMS INC   2020-01-01 2022-06-08         active    

In [45]:
import sys, os
target = os.path.expanduser('~/pip_libs')
%pip install --target={target} openpyxl
sys.path.append(target)

import importlib
importlib.invalidate_caches()
import openpyxl
print(openpyxl.__version__)

ERROR: Could not find a version that satisfies the requirement openpyxl (from versions: none)
ERROR: No matching distribution found for openpyxl
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'openpyxl'

In [43]:
vig.to_csv('padron_vigencias_2020_2024.csv', index=False)

maestra = vig.rename(columns={'ticker': 'symbol', 'comnam': 'security_name'})
maestra.to_excel('Lista Maestra Tickers ver 02.xlsx',
                 sheet_name='Vigencias', index=False)

ModuleNotFoundError: No module named 'openpyxl'

In [35]:
# 1. Conteo de permnos en cada etapa
print('vidas:      ', len(vidas), vidas['permno'].nunique())
print('vidas_enr:  ', len(vidas_enr), vidas_enr['permno'].nunique())
print('vig:        ', len(vig), vig['permno'].nunique())

vidas:       16828 11899
vidas_enr:   16828 11899
vig:         3626 3387


In [36]:
# 2. Los permnos con más tramos crudos (para ver qué se está repitiendo)
tmp = vidas_enr.groupby('permno').size().sort_values(ascending=False)
print(tmp.head(10))
print(tmp.describe())

permno
84302    8
16877    8
14051    7
25452    7
15423    7
16377    7
93126    7
18089    7
15904    6
22798    6
dtype: int64
count    11899.000000
mean         1.414236
std          0.777436
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          8.000000
dtype: float64


In [37]:
# 3. Anatomía del caso más repetido
pn = tmp.index[0]
print(vidas_enr[vidas_enr['permno'] == pn]
      [['permno', 'ticker', 'namedt', 'nameenddt', 'comnam', 'gvkey', 'cik']]
      .to_string(index=False))

 permno ticker     namedt  nameenddt                   comnam  gvkey        cik
  84302   AULT 2023-01-03 2023-05-17        AULT ALLIANCE INC 064163 0000896493
  84302   AULT 2023-05-18 2024-01-16        AULT ALLIANCE INC 064163 0000896493
  84302   AULT 2024-01-17 2024-09-09        AULT ALLIANCE INC 064163 0000896493
  84302    DPW 2019-08-06 2021-01-18       D P W HOLDINGS INC 064163 0000896493
  84302    DPW 2021-01-19 2021-12-12 AULT GLOBAL HOLDINGS INC 064163 0000896493
  84302   GPUS 2024-09-10 2024-11-24      HYPERSCALE DATA INC 064163 0000896493
  84302   GPUS 2024-11-25 2024-12-31      HYPERSCALE DATA INC 064163 0000896493
  84302   NILE 2021-12-13 2023-01-02     BITNILE HOLDINGS INC 064163 0000896493


In [38]:
# 1. Qué hay realmente en vidas
print(len(vidas), vidas['permno'].nunique())
for t in ['GME', 'ABNB', 'COIN', 'RDDT', 'BBBY']:
    sub = vidas[vidas['ticker'] == t]
    print(t, len(sub))
    if len(sub):
        print(sub[['permno', 'ticker', 'namedt', 'nameenddt', 'comnam']].to_string(index=False))

16828 11899
GME 2
 permno ticker     namedt  nameenddt            comnam
  89301    GME 2014-09-24 2021-03-28 GAMESTOP CORP NEW
  89301    GME 2021-03-29 2024-12-31 GAMESTOP CORP NEW
ABNB 1
 permno ticker     namedt  nameenddt     comnam
  20190   ABNB 2020-12-10 2024-12-31 AIRBNB INC
COIN 1
 permno ticker     namedt  nameenddt              comnam
  20892   COIN 2021-04-14 2024-12-31 COINBASE GLOBAL INC
RDDT 1
 permno ticker     namedt  nameenddt     comnam
  24876   RDDT 2024-03-21 2024-12-31 REDDIT INC
BBBY 1
 permno ticker     namedt  nameenddt                comnam
  77659   BBBY 1999-07-01 2023-05-02 BED BATH & BEYOND INC


In [39]:
# 2. Reconstruir desde la fuente en una variable NUEVA y comparar
vidas2 = db.raw_sql("""
    select permno, permco, namedt, nameenddt,
           ticker, comnam, ncusip, cusip,
           shrcd, exchcd, siccd, shrcls
    from crsp.stocknames
    where nameenddt >= '2020-01-01'
      and namedt   <= '2024-12-31'
      and exchcd in (1, 2, 3, 4)
    order by ticker, namedt
""", date_cols=['namedt', 'nameenddt'])

print(len(vidas2), vidas2['permno'].nunique())
print(vidas2[vidas2['ticker'].isin(['GME', 'ABNB', 'RDDT', 'BBBY'])]
      [['permno', 'ticker', 'namedt', 'nameenddt', 'comnam']].to_string(index=False))

16828 11899
 permno ticker     namedt  nameenddt                comnam
  20190   ABNB 2020-12-10 2024-12-31            AIRBNB INC
  77659   BBBY 1999-07-01 2023-05-02 BED BATH & BEYOND INC
  89301    GME 2014-09-24 2021-03-28     GAMESTOP CORP NEW
  89301    GME 2021-03-29 2024-12-31     GAMESTOP CORP NEW
  24876   RDDT 2024-03-21 2024-12-31            REDDIT INC


In [29]:
link = db.raw_sql("""
    select l.lpermno as permno, l.gvkey,
           l.linkdt, l.linkenddt,
           c.cik, c.gsector, c.ggroup, c.gind, c.conm
    from crsp.ccmxpf_lnkhist l
    join comp.company c on c.gvkey = l.gvkey
    where l.linktype in ('LU','LC')
      and l.linkprim in ('P','C')
""", date_cols=['linkdt', 'linkenddt'])


ProgrammingError: (psycopg2.errors.InsufficientPrivilege) permission denied for schema crsp_a_ccm

[SQL: 
    select l.lpermno as permno, l.gvkey,
           l.linkdt, l.linkenddt,
           c.cik, c.gsector, c.ggroup, c.gind, c.conm
    from crsp.ccmxpf_lnkhist l
    join comp.company c on c.gvkey = l.gvkey
    where l.linktype in ('LU','LC')
      and l.linkprim in ('P','C')
]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [8]:
link = db.raw_sql("""
    select l.lpermno as permno, l.gvkey,
           l.linkdt, l.linkenddt,
           c.cik, c.gsector, c.ggroup, c.gind, c.conm
    from crsp.ccmxpf_lnkhist l
    join comp.company c on c.gvkey = l.gvkey
    where l.linktype in ('LU','LC')
      and l.linkprim in ('P','C')
""", date_cols=['linkdt', 'linkenddt'])


ProgrammingError: (psycopg2.errors.InsufficientPrivilege) permission denied for schema crsp_a_ccm

[SQL: 
    select l.lpermno as permno, l.gvkey,
           l.linkdt, l.linkenddt,
           c.cik, c.gsector, c.ggroup, c.gind, c.conm
    from crsp.ccmxpf_lnkhist l
    join comp.company c on c.gvkey = l.gvkey
    where l.linktype in ('LU','LC')
      and l.linkprim in ('P','C')
]
(Background on this error at: https://sqlalche.me/e/20/f405)